## Librerías
---

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde
from scipy.optimize import minimize
from random import gauss, seed
from math import sqrt

## Simulación por Euler-Murayama
---

In [6]:
# --- Simulador ---
def simular_vasicek(x0, m, sigma, c, delta, n):
    seed(43); oo = sqrt(delta)
    l = np.zeros(n); l[0] = x0
    for i in range(n - 1):
        x = l[i]; t = 0
        while t < 1:
            w = oo * gauss(0, 1)
            x = x + c * (m - x) * delta + sigma * w
            t = t + delta
        l[i + 1] = x
    return l

## Método de máxima verosimilitud
---

In [7]:
# --- Log-verosimilitud con KDE VECTORIZADO ---
def log_likelihood_kde(params, datos, delta, ns_sim):
    """
    Optimizaciones:
    - Simulación vectorizada: se simulan los ns_sim caminos AL MISMO TIEMPO.
    - Se elimina el bucle interno `for j in range(ns_sim)`.
    - Se precalculan constantes fuera del bucle.
    """
    m, sigma, c = params
    if sigma <= 0 or c <= 0:
        return 1e12

    oo = sqrt(delta)
    n_sub = int(round(1 / delta))  # pasos internos por intervalo
    n = len(datos)
    suma_log = 0.0

    for i in range(n - 1):
        # Simulación VECTORIZADA: ns_sim caminos en paralelo dentro de un array
        # Forma: (ns_sim, n_sub)
        Z = np.random.normal(0, oo, size=(ns_sim, n_sub))
        X_sim = np.full(ns_sim, datos[i])
        for k in range(n_sub):
            X_sim = X_sim + c * (m - X_sim) * delta + sigma * Z[:, k]

        # KDE sobre los ns_sim valores finales
        pdf = gaussian_kde(X_sim)
        valor = pdf(datos[i + 1])[0]

        if valor < 1e-200:
            valor = 1e-200
        suma_log += np.log(valor)

    return -suma_log

# --- Callback ---
estado = {'iter': 0}
def callback_progreso(xk):
    estado['iter'] += 1
    val = log_likelihood_kde(xk, datos_obs, delta, ns_sim=500)
    print(f"Iter {estado['iter']:3d} | m={xk[0]:.6f} | sigma={xk[1]:.6f} | "
          f"c={xk[2]:.6f} | -LL={val:.4f}")

## Resultados
---

In [8]:
# --- Ejecución ---
m_real, sigma_real, c_real = 0.05, 0.02, 1.5
delta = 0.01
datos_obs = simular_vasicek(0.02, m_real, sigma_real, c_real, delta, 100)  # n=100 para que sea rápido

x0_est = np.array([0.04, 0.015, 1.2])

print("=" * 78)
print("MLE con KDE vectorizado + Nelder-Mead")
print("=" * 78)

resultado = minimize(
    log_likelihood_kde,
    x0_est,
    args=(datos_obs, delta, 500),  # ns_sim=500 (antes era 1000)
    method='Nelder-Mead',
    callback=callback_progreso,
    options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 200}
)

print("=" * 78)
print(f"Estimados: m={resultado.x[0]:.6f}, sigma={resultado.x[1]:.6f}, c={resultado.x[2]:.6f}")

MLE con KDE vectorizado + Nelder-Mead


Iter   1 | m=0.042000 | sigma=0.015000 | c=1.200000 | -LL=-307.8857
Iter   2 | m=0.042000 | sigma=0.015000 | c=1.200000 | -LL=-309.0254
Iter   3 | m=0.045556 | sigma=0.014833 | c=1.100000 | -LL=-318.7521
Iter   4 | m=0.045556 | sigma=0.014833 | c=1.100000 | -LL=-318.5260
Iter   5 | m=0.045556 | sigma=0.014833 | c=1.100000 | -LL=-317.5185
Iter   6 | m=0.049342 | sigma=0.014198 | c=1.078519 | -LL=-319.4633
Iter   7 | m=0.049342 | sigma=0.014198 | c=1.078519 | -LL=-320.5884
Iter   8 | m=0.049342 | sigma=0.014198 | c=1.078519 | -LL=-319.2448
Iter   9 | m=0.049342 | sigma=0.014198 | c=1.078519 | -LL=-321.0438
Iter  10 | m=0.049342 | sigma=0.014198 | c=1.078519 | -LL=-318.8900
Iter  11 | m=0.049342 | sigma=0.014198 | c=1.078519 | -LL=-320.1447
Iter  12 | m=0.049342 | sigma=0.014198 | c=1.078519 | -LL=-321.5211
Iter  13 | m=0.048810 | sigma=0.014505 | c=1.099684 | -LL=-321.0368
Iter  14 | m=0.048717 | sigma=0.014238 | c=1.115527 | -LL=-320.6901
Iter  15 | m=0.049147 | sigma=0.014259 | c=1.096